# Scientist track: full A100 jobs

GPU results: **PENDING (A100 Colab)**. These cells have not been run on a GPU here.

Unpack a snapshot of the current workspace (code, configs, tests, benchmarks, data/v0) at `/content/labbench-posttrain`. Exclude data/raw, .git, .venv and outputs. Do not use an older remote copy. Run cells top to bottom on an A100. Dependencies are installed into a venv that reuses Colab CUDA torch. No HuggingFace uploads.

Each method is an independent baseline initialized from the model in its config. DPO/GRPO do not automatically continue the SFT adapter. For a sequential experiment, merge the SFT adapter locally and explicitly override model=<merged directory>. T3 assay calibration and live blind experiments remain TODO. Long jobs may exceed a single Colab session; retain outputs on Drive if needed.

In [ ]:
from pathlib import Path
import os, subprocess, sys
repo = Path('/content/labbench-posttrain')
assert (repo / 'data/v0/records.parquet').is_file(), 'Unpack the current repo snapshot with data/v0 at /content/labbench-posttrain first.'
os.chdir(repo)
subprocess.run([sys.executable, '-m', 'venv', '--system-site-packages', '.venv'], check=True)
PY = str(repo / '.venv/bin/python')
subprocess.run([PY, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)

In [ ]:
import torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > A100.'
assert 'A100' in torch.cuda.get_device_name(0), torch.cuda.get_device_name(0)
torch.backends.cuda.matmul.allow_tf32 = True
os.environ['NVIDIA_TF32_OVERRIDE'] = '1'
print(subprocess.run(['nvidia-smi'], check=True, capture_output=True, text=True).stdout)
def run(*args):
    subprocess.run([PY, *args], check=True)

In [ ]:
# Verify data contracts and refresh reports before long jobs.
run('-m', 'pytest', '-q')
run('scripts/assay_quality.py')
run('scripts/benchmark_link.py')

## Representations

In [ ]:
run('scientist/train_encoders.py', 'configs/scientist/encoders_a100.yaml')

## Modality/text alignment

In [ ]:
run('scientist/train_align.py', 'configs/scientist/align_a100.yaml')

## Molecule soft-token projector + LoRA

In [ ]:
run('scientist/train_projector.py', 'configs/scientist/projector_a100.yaml')

## SFT with LoRA

In [ ]:
run('scripts/train_sft.py', 'configs/scientist/sft_a100.yaml')

## DPO

In [ ]:
run('scripts/train_pref.py', 'configs/scientist/dpo_a100.yaml')

## Reward model

In [ ]:
run('scripts/train_rm.py', 'configs/scientist/rm_a100.yaml')

## GRPO with T1/T2 rewards and KL

In [ ]:
run('scripts/train_grpo.py', 'configs/scientist/grpo_a100.yaml')

## Teacher/student distillation

In [ ]:
run('scientist/distill.py', 'configs/scientist/distill_a100.yaml')

## Held-out SFT evaluation

In [ ]:
run('scripts/eval_llm.py', '--model', 'Qwen/Qwen2.5-7B-Instruct', '--adapter', 'outputs/a100/scientist_sft/adapter', '--n', '200', '--out', 'outputs/a100/eval_scientist_sft.json')

## Time split and blind reservation (offline rehearsal)

In [ ]:
# Builds a separate dataset; full jobs above use the checked-in group split.
run('scripts/build_dataset.py', '--out', 'outputs/time_split', '--split-mode', 'time', '--val-year', '2009', '--test-year', '2011', '--prospective-source', 'platero_cytation3') if Path('data/interim').is_dir() else print('Optional split rebuild needs data/interim; do not download or modify data/raw.')
# To train on this rebuild, generate tasks with --data-dir outputs/time_split, then override train_file/eval_file.
# prospective_blind.parquet stays out of routine task generation; lab staff must seal future results.

In [ ]:
import json
for path in sorted(Path('outputs/a100').glob('**/profile/summary.json')):
    print(path, json.loads(path.read_text()))